# Week 4 — Supervised Learning Model Implementation
**Student Name:** Sindhu Patil | **Internship:** Data Science

## 1. Introduction
Supervised machine learning trains algorithms on historical data paired with ground-truth target labels to learn predictive mappings.

## 2. Problem Statement
Customer churn in telecommunications leads to revenue loss. Predicting subscriber churn prior to cancellation enables proactive retention interventions.

## 3. Objective
Build, evaluate, and interpret a binary classification pipeline using Logistic Regression on `data/processed/cleaned_telco_churn.csv`.

## 4. Dataset Overview
Loading cleaned dataset (7,043 rows, 21 columns). `customerID` identifier is strictly excluded.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from src.modeling import build_preprocessing_pipeline, build_model_pipeline, evaluate_cv_performance, evaluate_test_performance, extract_logistic_coefficients
from src.visualization import plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve, plot_threshold_analysis, plot_model_comparison, plot_top_coefficients

df = pd.read_csv('../data/processed/cleaned_telco_churn.csv')
print('Cleaned Dataset Shape:', df.shape)

## 5. Target Variable Preparation (`Churn`)
Converting binary target: `Yes` -> 1 (Churned), `No` -> 0 (Retained). Target is imbalanced (26.54% positive class).

In [ ]:
X = df.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)
print('Target class counts:\n', y.value_counts(normalize=True))

## 6. Feature Selection & Feature Engineering
Categorizing 3 continuous numerical features (`tenure`, `MonthlyCharges`, `TotalCharges`) and 15 categorical predictors.

In [ ]:
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in X.columns if c not in numerical_features]
print('Numerical features:', numerical_features)
print('Categorical features:', categorical_features)

## 7. Train-Test Split (80/20 Stratified)
Splitting data into 5,634 training rows and 1,409 testing rows. Stratification preserves the 26.54% churn ratio in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('X_train shape:', X_train.shape, '| X_test shape:', X_test.shape)

## 8. Preprocessing Pipeline (`ColumnTransformer`)
`StandardScaler` standardizes continuous features, while `OneHotEncoder` transforms categorical predictors. Encapsulated inside `Pipeline` to prevent data leakage.

In [ ]:
preprocessor = build_preprocessing_pipeline(numerical_features, categorical_features)

## 9. Primary Model — Logistic Regression
Constructing `Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=1000, random_state=42))])`.

In [ ]:
lr_pipeline = build_model_pipeline(LogisticRegression(max_iter=1000, random_state=42), preprocessor)

## 10. 5-Fold Stratified Cross-Validation

In [ ]:
cv_summary, cv_results = evaluate_cv_performance(lr_pipeline, X, y, cv_folds=5)
for metric, vals in cv_summary.items():
    print(f'{metric.upper():12s}: Mean = {vals["mean"]:.4f} +/- {vals["std"]:.4f}')

## 11. Model Training & Test Set Evaluation

In [ ]:
metrics_lr, y_pred_lr, y_prob_lr, cm_lr = evaluate_test_performance(lr_pipeline, X_train, X_test, y_train, y_test, 'Logistic Regression')
print('Test Set Performance (Logistic Regression):\n', metrics_lr)

## 12. Confusion Matrix Analysis

In [ ]:
plot_confusion_matrix(cm_lr, ['Retained (No)', 'Churned (Yes)'], '../outputs/figures/week4/confusion_matrix.png')
plt.figure(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
plt.title('Confusion Matrix (Test Set)')
plt.show()

## 13. Classification Report

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_lr, target_names=['Retained', 'Churned']))

## 14. ROC Curve & AUC Evaluation

In [ ]:
plot_roc_curve(y_test, {'Logistic Regression': (y_prob_lr, metrics_lr['ROC_AUC'])}, '../outputs/figures/week4/roc_curve.png')

## 15. Precision-Recall Curve

In [ ]:
plot_precision_recall_curve(y_test, {'Logistic Regression': (y_prob_lr, metrics_lr['Average_Precision'])}, '../outputs/figures/week4/precision_recall_curve.png')

## 16. Classification Threshold Analysis

In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)
precisions = [precision_score(y_test, y_prob_lr >= t, zero_division=0) for t in thresholds]
recalls = [recall_score(y_test, y_prob_lr >= t, zero_division=0) for t in thresholds]
f1s = [f1_score(y_test, y_prob_lr >= t, zero_division=0) for t in thresholds]
plot_threshold_analysis(thresholds, precisions, recalls, f1s, '../outputs/figures/week4/threshold_analysis.png')

## 17. Feature Coefficient Interpretation

In [ ]:
df_coef = extract_logistic_coefficients(lr_pipeline, numerical_features, categorical_features)
plot_top_coefficients(df_coef, top_n=12, filename='../outputs/figures/week4/top_logistic_coefficients.png')
print(df_coef.head(10))

## 18. Model Comparison (Logistic Regression vs Random Forest)

In [ ]:
rf_pipeline = build_model_pipeline(RandomForestClassifier(n_estimators=100, random_state=42), preprocessor)
metrics_rf, y_pred_rf, y_prob_rf, cm_rf = evaluate_test_performance(rf_pipeline, X_train, X_test, y_train, y_test, 'Random Forest')
comp_df = pd.DataFrame([metrics_lr, metrics_rf])
plot_model_comparison(comp_df, '../outputs/figures/week4/model_comparison.png')
print(comp_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC']])

## 19. Final Model Selection Reasoning
Logistic Regression was selected over Random Forest because it achieved superior Recall (**0.5588** vs 0.4759), higher F1-Score (**0.6040** vs 0.5370), and higher ROC-AUC (**0.8419** vs 0.8179), while offering clear coefficient interpretability.

## 20. Strengths & Limitations
- **Strengths:** 0% data leakage via sklearn Pipeline, balanced evaluation across Recall/F1/ROC-AUC, clear feature coefficient rankings.
- **Limitations:** Imbalanced dataset (~26.5% positive class) limits un-tuned Recall to ~55.9%.

## 21. Conclusion
Logistic Regression established a robust baseline churn model with 80.55% accuracy and 0.8419 ROC-AUC, identifying short tenure, month-to-month contracts, and fiber optic service as primary churn drivers.